# Visualization of simulation

In [35]:
# Load the data from the pickle file
from IPython.display import HTML
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
# Load the pickled DataFrame
with open("simulation_data.pkl", "rb") as f:
    df = pickle.load(f)

In [36]:
%matplotlib notebook

# Load your simulation data
with open("simulation_data.pkl", "rb") as f:
    df = pickle.load(f)

# Separate bees and plants
bee_df = df[df["Agent Type"] == "bee"]
plant_df = df[(df.index.get_level_values("Step") == 0) & (df["Agent Type"] == "plant")]

# Extract plant coordinates once
plant_x = [pos[0] for pos in plant_df["Bee Positions"]]
plant_y = [pos[1] for pos in plant_df["Bee Positions"]]

# Get only steps that contain bee data
steps = sorted(bee_df.index.get_level_values("Step").unique())

# Prepare figure
fig, ax = plt.subplots(figsize=(6, 6))

# Static plot for plant locations
ax.scatter(plant_x, plant_y, c='green', s=30, label="Plants")

# Create bee scatter object
scat = ax.scatter([], [], s=30, c='orange', label="Bees")

# Set axis limits based on all bee + plant positions
all_bee_positions = bee_df["Bee Positions"].tolist()
all_x = [pos[0] for pos in all_bee_positions] + plant_x
all_y = [pos[1] for pos in all_bee_positions] + plant_y
ax.set_xlim(min(all_x) - 1, max(all_x) + 1)
ax.set_ylim(min(all_y) - 1, max(all_y) + 1)
ax.set_title("Bee Movement Relative to Plants")
ax.set_xlabel("X Position")
ax.set_ylabel("Y Position")
ax.legend()
ax.grid(True)

# Init function: empty scatter array with shape (0,2)
def init():
    scat.set_offsets(np.empty((0, 2)))
    return scat,

# Update function for each frame (step)
def update(frame):
    # Safely get all bees at this step
    if frame not in bee_df.index.get_level_values("Step"):
        scat.set_offsets(np.empty((0, 2)))
    else:
        current_bees = bee_df.loc[frame]
        if isinstance(current_bees, pd.Series):  # Only one bee
            coords = [current_bees["Bee Positions"]]
        else:  # Multiple bees
            coords = current_bees["Bee Positions"].tolist()
        scat.set_offsets(coords)
    ax.set_title(f"Step {frame} - Bee Movement Relative to Plants")
    return scat,



# Create animation 
anim = FuncAnimation(fig, update, frames=steps, init_func=init,
                     blit=True, interval=300, repeat=False)

HTML(anim.to_jshtml())

<IPython.core.display.Javascript object>